# NEXUS: CatBoost with risk-adjusted entity history

Version 2 fixes target leakage found in the first Kaggle run: expected_oof and residual_oof are history-work columns and cannot enter CatBoost features. Forward nuisance probabilities create matured residual histories. A bounded logistic correction is fitted on earlier frozen six-month snapshots.

Enable a Kaggle GPU and attach the seven-file competition dataset. Run all cells in order. The final CSV and diagnostics are saved in `/kaggle/working`. The model is scratch-trained using only supplied data. The July–December 2024 fold has been repeatedly examined and is reported as a development diagnostic. No result is claimed until this notebook runs.


## 1. Competition feature implementation

Embedded from `optiml/features.py`; the notebook has no repository dependency.


In [ ]:
"""Target-independent joins and features. Raw CSV files are never modified."""
from pathlib import Path
import numpy as np
import pandas as pd

VERSION = 'strategy-v1'
KEYS = ['order_line_id', 'order_id', 'customer_id', 'product_id', 'store_id']
CATEGORY_TYPOS = {
    'electronicsaccessories': 'electronics_accessories', 'fashion_acessories': 'fashion_accessories',
    'food_beerages': 'food_beverages', 'furnture': 'furniture', 'garde_diy': 'garden_diy',
    'healthbeauty': 'health_beauty', 'home_ecor': 'home_decor', 'kitchendining': 'kitchen_dining',
    'office_upplies': 'office_supplies', 'sports_eisure': 'sports_leisure',
    'technlogy': 'technology', 'toys_ifts': 'toys_gifts',
}

def normalize(s):
    return s.astype('string').str.strip().str.lower().str.replace(r'[\s-]+', '_', regex=True)

def ratio(a, b):
    return a.div(b.where(b != 0)).replace([np.inf, -np.inf], np.nan)

def dimensions(data_dir):
    root = Path(data_dir)
    customer = pd.read_csv(root / 'customers.csv')
    customer['record_updated_at'] = pd.to_datetime(customer.record_updated_at, errors='raise')
    customer = customer.sort_values(['customer_id', 'record_updated_at'], kind='stable').drop_duplicates('customer_id', keep='last')
    # Supplied-snapshot policy for region; mutable loyalty and update time are excluded.
    customer = customer.drop(columns=['record_updated_at', 'loyalty_tier'])
    customer['region'] = normalize(customer.region)
    product = pd.read_csv(root / 'products.csv').drop(columns=['launch_date', 'review_count'])
    product['category'] = normalize(product.category).replace(CATEGORY_TYPOS)
    store = pd.read_csv(root / 'stores.csv').drop(columns=['seller_rating', 'manager_tenure_years'])
    store['region'] = normalize(store.region)
    calendar = pd.read_csv(root / 'calendar.csv')
    calendar['week_start'] = pd.to_datetime(calendar.week_start)
    result = []
    for frame, key, prefix in [(customer, 'customer_id', 'customer'), (product, 'product_id', 'product'), (store, 'store_id', 'store')]:
        if frame[key].duplicated().any():
            raise ValueError(f'Duplicate dimension key: {key}')
        result.append(frame.rename(columns={c: f'{prefix}_{c}' for c in frame if c != key}))
    result.append(calendar)
    return result

def build_features(raw, dims):
    df = raw.copy()
    if df.order_line_id.duplicated().any():
        raise ValueError('Duplicate order_line_id')
    original_ids = df.order_line_id.copy()
    ts = pd.to_datetime(df.order_ts, errors='raise')
    # Raw wall-clock timestamps: no unjustified timezone conversion.
    # Assign every order to the earliest line time so an order cannot straddle
    # a temporal fold when line timestamps differ.
    df['_order_time'] = ts.groupby(df.order_id).transform('min')
    df['_month'] = df['_order_time'].dt.strftime('%Y-%m')
    df['order_timestamp_span_hours'] = (df['_order_time'] - ts.groupby(df.order_id).transform('min')).dt.total_seconds() / 3600
    for dim, key in zip(dims[:3], ['customer_id', 'product_id', 'store_id']):
        if not df[key].isin(dim[key]).all():
            raise ValueError(f'Unmatched {key}')
        df = df.merge(dim, on=key, how='left', sort=False, validate='many_to_one')
    df['week_start'] = ts.dt.normalize() - pd.to_timedelta(ts.dt.dayofweek, unit='D')
    df = df.merge(dims[3], on='week_start', how='left', sort=False, validate='many_to_one', indicator='_calendar_join')
    if not df['_calendar_join'].eq('both').all():
        raise ValueError('Unmatched calendar week')
    if not df.order_line_id.reset_index(drop=True).equals(original_ids.reset_index(drop=True)):
        raise AssertionError('Join changed row order or count')
    df = df.drop(columns=['_calendar_join', 'week_start'])
    df['month_of_year'] = ts.dt.month
    df['day_of_week'] = ts.dt.dayofweek
    df['hour'] = ts.dt.hour
    df['weekend'] = (ts.dt.dayofweek >= 5).astype('int8')
    for field, out in [('customer_signup_date', 'customer_tenure_days'), ('store_opened_date', 'store_age_days')]:
        age = (ts - pd.to_datetime(df[field], errors='raise')).dt.total_seconds() / 86400
        df[out + '_invalid'] = (age < 0).astype('int8')
        df[out] = age.where(age >= 0)
        df = df.drop(columns=field)
    df['customer_store_region_match'] = (df.customer_region == df.store_region).astype('int8')
    df['line_gross_candidate'] = df.unit_price * df.quantity
    df['price_to_list_ratio'] = ratio(df.unit_price, df.product_list_price)
    df['amount_to_gross_ratio'] = ratio(df.order_value, df.line_gross_candidate)
    df['amount_gross_residual'] = df.order_value - df.line_gross_candidate
    df['freight_per_unit'] = ratio(df.freight_cost, df.quantity)
    df['freight_to_gross_ratio'] = ratio(df.freight_cost, df.line_gross_candidate)
    df['implied_discount_residual'] = (1 - df.price_to_list_ratio) - df.discount
    df['product_volume_cm3'] = df.product_length_cm * df.product_width_cm * df.product_height_cm
    df['product_density_kg_cm3'] = ratio(df.product_weight_kg, df.product_volume_cm3)
    df['line_weight_kg'] = df.product_weight_kg * df.quantity
    df['coupon_present'] = df.coupon_code.notna().astype('int8')
    for c in ['payment_method', 'ship_mode', 'discount', 'freight_cost', 'session_clicks', 'product_weight_kg', 'customer_age_band', 'store_size_sqm']:
        df[c + '_missing'] = df[c].isna().astype('int8')
    group = df.groupby('order_id', sort=False)
    df['basket_lines'] = group.order_line_id.transform('size')
    if not df.basket_lines.eq(df.items_in_order).all():
        raise ValueError('Incomplete orders or inconsistent items_in_order')
    df['basket_units'] = group.quantity.transform('sum')
    df['basket_gross_candidate'] = group.line_gross_candidate.transform('sum')
    df['basket_categories'] = group.product_category.transform('nunique')
    df['basket_products'] = group.product_id.transform('nunique')
    df['basket_stores'] = group.store_id.transform('nunique')
    df['basket_mean_discount'] = group.discount.transform('mean')
    df['line_basket_value_share'] = ratio(df.line_gross_candidate, df.basket_gross_candidate)
    df['line_basket_units_share'] = ratio(df.quantity, df.basket_units)
    df['relative_line_position'] = ratio(df.line_no, df.basket_lines)
    for c in ['unit_price', 'quantity', 'order_value', 'freight_cost', 'basket_gross_candidate', 'basket_units', 'product_volume_cm3']:
        df['log1p_' + c] = np.log1p(df[c].where(df[c] >= 0))
    df = df.drop(columns=['order_ts', 'coupon_code', 'store_channel'])
    df['banner_id'] = df.banner_id.astype('string')
    for c in df.select_dtypes(include='number'):
        if c != 'returned':
            df[c] = df[c].astype('float32')
    return df

def feature_columns(frame, include_promo=False):
    excluded = {'returned', 'order_line_id', 'order_id', '_order_time', '_month'}
    if not include_promo:
        excluded.add('promo_channel_code')
    return [c for c in frame if c not in excluded]

def feature_manifest(frame, include_promo=False):
    """Describe feature provenance for reproducible local experiments."""
    excluded = {'returned', 'order_line_id', 'order_id', '_order_time', '_month'}
    if not include_promo:
        excluded.add('promo_channel_code')
    rows = []
    for name in frame.columns:
        if name in excluded:
            continue
        source = 'derived'
        if name in {'customer_id', 'product_id', 'store_id'}:
            source = 'dimension key'
        elif name in {'channel', 'payment_method', 'ship_mode', 'unit_price', 'quantity',
                      'discount', 'freight_cost', 'order_value', 'line_no', 'items_in_order',
                      'gift_wrap', 'promo_channel_code', 'return_window_days', 'warehouse_zone',
                      'picker_shift', 'banner_id', 'session_clicks', 'device_type', 'tz_hint'}:
            source = 'transaction'
        elif name.endswith('_past_rate') or name.endswith('_past_count'):
            source = 'past labels; prior months only'
        rows.append({
            'feature': name,
            'source': source,
            'availability': 'before or at order time',
            'uses_labels': name.endswith('_past_rate') or name.endswith('_past_count'),
            'missing_behavior': 'native missing value' if frame[name].isna().any() else 'not observed',
        })
    return rows

def encode_categories(train, other, columns):
    """Fit vocabulary only on fitting rows; unseen categories become missing."""
    x = train[columns].copy()
    z = other[columns].copy()
    levels = {}
    for c in columns:
        if not pd.api.types.is_numeric_dtype(x[c]):
            values = sorted(x[c].dropna().astype(str).unique().tolist())
            levels[c] = values
            x[c] = pd.Categorical(x[c], categories=values)
            z[c] = pd.Categorical(z[c].where(z[c].isin(values)), categories=values)
    return x, z, levels


## 2. Load data and define forward evaluation

Orders are assigned to their latest timestamp. Each forecast spans six months. The 75-day delay applies to matured history; a separate full supervised-fit embargo is not claimed here. Selection uses earlier folds only.


In [ ]:
"""Embedded into the three standalone Kaggle challenger notebooks by build_challenger_notebooks.py."""
import gc, hashlib, json, os, platform, time
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier, Pool

SEED = 42
OUT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
NEEDED = {'train.csv','test.csv','customers.csv','products.csv','stores.csv','calendar.csv','sample_submission.csv'}
roots = [p.parent for p in Path('/kaggle/input').rglob('train.csv') if NEEDED.issubset(q.name for q in p.parent.iterdir())] if Path('/kaggle/input').exists() else []
local = Path('nexus-opti-ml-2026-operation-insight')
if local.exists() and NEEDED.issubset(q.name for q in local.iterdir()): roots.append(local)
assert len(roots) == 1, f'Attach one NEXUS dataset; found {roots}'
DATA = roots[0]
sample = pd.read_csv(DATA/'sample_submission.csv')
raw_train = pd.read_csv(DATA/'train.csv', low_memory=False)
raw_test = pd.read_csv(DATA/'test.csv', low_memory=False)
assert raw_train.order_line_id.is_unique and raw_test.order_line_id.is_unique
assert set(raw_train.order_id).isdisjoint(raw_test.order_id)
assert sample.columns.tolist() == ['order_line_id','returned']
assert set(sample.order_line_id) == set(raw_test.order_line_id)
dims = dimensions(DATA)
train = build_features(raw_train, dims)
test = build_features(raw_test, dims)
# Conservative split convention: assign an order to its latest line timestamp.
for frame, raw in ((train,raw_train),(test,raw_test)):
    frame['_order_time'] = pd.to_datetime(raw.order_ts).groupby(raw.order_id).transform('max').to_numpy()
    frame['_month'] = frame._order_time.dt.strftime('%Y-%m')
    assert frame.order_id.groupby(frame.order_id).transform('size').eq(frame.basket_lines).all()
del raw_train, raw_test, dims
gc.collect()
assert len(train) == 1400005 and len(test) == 479995

BASE_DROP = {'returned','customer_id','product_id','store_id','product_supplier_code','banner_id',
             'promo_channel_code','cpi','fuel_price','temperature_f','unemployment',
             'fuel_surcharge','customer_loyalty_tier','product_review_count',
             'store_seller_rating','store_manager_tenure_years',
             # These columns are created only for the risk-adjusted history
             # calculation. residual_oof contains the row's own label.
             'expected_oof','residual_oof'}
CALENDAR = {'month_sin','month_cos','weekday_sin','weekday_cos','holiday_flag'}
def add_calendar(frame):
    out = frame.copy()
    dt = out._order_time
    out['month_sin'] = np.sin(2*np.pi*(dt.dt.month-1)/12).astype('float32')
    out['month_cos'] = np.cos(2*np.pi*(dt.dt.month-1)/12).astype('float32')
    out['weekday_sin'] = np.sin(2*np.pi*dt.dt.dayofweek/7).astype('float32')
    out['weekday_cos'] = np.cos(2*np.pi*dt.dt.dayofweek/7).astype('float32')
    out['holiday_flag'] = pd.to_numeric(out.holiday_flag, errors='coerce').fillna(0).astype('float32') if 'holiday_flag' in out else 0.0
    return out
train, test = add_calendar(train), add_calendar(test)

def features(frame, seasonal=True):
    cols = [c for c in feature_columns(frame, include_promo=False)
            if c not in BASE_DROP and not c.endswith('_age_days') and c not in {'customer_tenure_days','store_age_days'}]
    if not seasonal: cols = [c for c in cols if c not in CALENDAR]
    assert not {'returned','expected_oof','residual_oof'} & set(cols), 'Target-derived column entered model features'
    return cols

def matrix(fit, other, cols):
    x, z, levels = encode_categories(fit, other, cols)
    cats = list(levels)
    for c in cats:
        x[c] = x[c].astype('string').fillna('__MISSING__').astype(str)
        z[c] = z[c].astype('string').fillna('__MISSING__').astype(str)
    for c in set(cols)-set(cats):
        x[c] = pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
        z[c] = pd.to_numeric(z[c],errors='coerce').replace([np.inf,-np.inf],np.nan).astype('float32')
    return x, z, cats

def anchor(fit, other, iterations=1800, seed=SEED, extra_cols=()):
    cols = features(fit) + list(extra_cols)
    assert not {'returned','expected_oof','residual_oof'} & set(cols)
    assert set(cols) <= set(other.columns), f'Inference columns missing: {set(cols)-set(other.columns)}'
    x,z,cats = matrix(fit,other,cols)
    m = CatBoostClassifier(iterations=iterations, depth=5, learning_rate=.05,
          l2_leaf_reg=5, max_ctr_complexity=1, loss_function='Logloss',
          task_type='GPU', devices='0', random_seed=seed, verbose=False,
          allow_writing_files=False)
    m.fit(Pool(x,label=fit.returned,cat_features=cats))
    p = m.predict_proba(Pool(z,cat_features=cats))[:,1].astype('float32')
    return p,m,cols

def folds():
    for origin in ('2023-07-01','2024-01-01','2024-07-01'):
        start = pd.Timestamp(origin); end = start + pd.DateOffset(months=6)
        fit = train.loc[train._order_time < start].copy()
        valid = train.loc[(train._order_time >= start)&(train._order_time < end)].copy()
        if not len(valid): continue
        assert fit._order_time.max() < valid._order_time.min()
        assert set(fit.order_id).isdisjoint(valid.order_id)
        yield origin,fit,valid

def safe_auc(y,p):
    y=np.asarray(y); p=np.asarray(p)
    return float(roc_auc_score(y,p)) if len(np.unique(y))==2 else None

def diagnostics(valid, pred):
    y=valid.returned.to_numpy(); p=np.asarray(pred)
    out={'pooled':safe_auc(y,p)}
    age=(valid._order_time.dt.year-valid._order_time.min().year)*12+valid._order_time.dt.month-valid._order_time.min().month
    groups={'early_horizon':age<3,'late_horizon':age>=3,'large_basket':valid.basket_lines>=51}
    for c in ('channel',):
        if c in valid:
            groups.update({f'{c}:{v}':valid[c].eq(v) for v in valid[c].dropna().unique()})
    for month in sorted(valid._month.unique()): groups[f'month:{month}']=valid._month.eq(month)
    for name,mask in groups.items():
        a=np.asarray(mask,dtype=bool); out[name]={'rows':int(a.sum()),'auc':safe_auc(y[a],p[a])}
    return out

def paired_order_ci(valid, a, b, n=200, seed=SEED):
    rng=np.random.default_rng(seed); ids=valid.order_id.to_numpy()
    unique,inv=np.unique(ids,return_inverse=True)
    gains=[]; y=valid.returned.to_numpy()
    for _ in range(n):
        counts=np.bincount(rng.integers(len(unique),size=len(unique)),minlength=len(unique))
        weights=counts[inv]
        if len(np.unique(y[weights>0]))==2:
            gains.append(float(roc_auc_score(y,b,sample_weight=weights)-
                               roc_auc_score(y,a,sample_weight=weights)))
    return np.quantile(gains,[.025,.5,.975]).tolist()

def save_prediction(name, frame, p):
    pd.DataFrame({'order_line_id':frame.order_line_id.to_numpy(),'returned':np.asarray(p)}).to_csv(OUT/name,index=False)

def submission(name,p):
    values=pd.DataFrame({'order_line_id':test.order_line_id,'returned':np.asarray(p,dtype='float64')})
    sub=sample[['order_line_id']].merge(values,on='order_line_id',how='left',sort=False,validate='one_to_one')
    assert len(sub)==479995 and sub.order_line_id.equals(sample.order_line_id)
    assert np.isfinite(sub.returned).all() and sub.returned.between(0,1).all()
    path=OUT/name; sub.to_csv(path,index=False)
    check=pd.read_csv(path)
    assert check.order_line_id.equals(sample.order_line_id)
    print('Saved',path,'SHA256',hashlib.sha256(path.read_bytes()).hexdigest())
    return path

def report(name, data):
    data['versions']={'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}
    (OUT/name).write_text(json.dumps(data,indent=2,default=str))

print('Data',DATA,'Rows',len(train),len(test),'Output',OUT)


## 3. Train, compare, and write the submission

The notebook saves fold predictions, cohort diagnostics, an order-bootstrap comparison, package versions, and a checksummed submission CSV.


In [ ]:
"""Risk-adjusted residual history; embedded into the standalone notebook."""
from sklearn.linear_model import LogisticRegression

DELAY = pd.Timedelta(days=75)
WINDOW = pd.Timedelta(days=365)
ENTITY = {'customer':'customer_id','product':'product_id','store':'store_id'}
PARENT = {'customer':'customer_customer_type','product':'product_category','store':'store_store_format'}

def nuisance_oof(frame, use_promo=True):
    """Predict historical blocks only with models fitted before each block."""
    frame=frame.sort_values('_order_time').copy()
    out=pd.Series(np.nan,index=frame.index,dtype='float32')
    periods=frame._order_time.dt.to_period('Q')
    for quarter in sorted(periods.unique()):
        block=frame.loc[periods.eq(quarter)]
        fit=frame.loc[frame._order_time < block._order_time.min()-DELAY]
        if len(fit)<25000: continue
        cols=features(fit)
        if use_promo and 'promo_channel_code' in fit: cols.append('promo_channel_code')
        x,z,cats=matrix(fit,block,cols)
        model=CatBoostClassifier(iterations=500,depth=4,learning_rate=.06,
            loss_function='Logloss',task_type='GPU',devices='0',random_seed=SEED,
            verbose=False,allow_writing_files=False)
        model.fit(Pool(x,label=fit.returned,cat_features=cats))
        out.loc[block.index]=model.predict_proba(Pool(z,cat_features=cats))[:,1].astype('float32')
        del x,z,model
        gc.collect()
    return out

train['expected_oof']=nuisance_oof(train)
train['residual_oof']=train.returned-train.expected_oof
print('Forward nuisance coverage',train.expected_oof.notna().mean())
assert set(features(train)) <= set(test.columns), 'Final test is missing a model feature'
assert not {'returned','expected_oof','residual_oof'} & set(features(train))

def residual_features(history, query, origin):
    """Freeze one 365-day matured snapshot for the entire six-month forecast."""
    cutoff=pd.Timestamp(origin)-DELAY
    hist=history.loc[(history._order_time < cutoff)&
                     (history._order_time >= cutoff-WINDOW)&
                     history.residual_oof.notna()].copy()
    assert len(hist)==0 or hist._order_time.max()<query._order_time.min()
    out=query.copy()
    for name,key in ENTITY.items():
        if key not in hist or key not in out: continue
        parent=PARENT[name]
        if parent not in hist or parent not in out: parent='channel'
        parent_mean=hist.groupby(parent,dropna=False).residual_oof.mean()
        fallback=out[parent].map(parent_mean).fillna(0).to_numpy()
        summary=hist.groupby(key,dropna=False).agg(
            residual_sum=('residual_oof','sum'), lines=('residual_oof','size'),
            orders=('order_id','nunique'), first=('_order_time','min'),
            last=('_order_time','max'))
        # Distinct-order shrinkage and age decay prevent line-heavy, stale histories dominating.
        n=out[key].map(summary.orders).fillna(0).to_numpy()
        cnt=out[key].map(summary.lines).fillna(0).to_numpy()
        sums=out[key].map(summary.residual_sum).fillna(0).to_numpy()
        last=pd.to_datetime(out[key].map(summary['last']))
        age=(cutoff-last).dt.days.fillna(365).clip(lower=0).to_numpy()
        raw=np.divide(sums,np.maximum(cnt,1))
        shrink=n/(n+20)*np.exp(-age/365)
        out[f'{name}_residual_effect']=np.clip(shrink*raw+(1-shrink)*fallback*.2,-.3,.3).astype('float32')
        out[f'{name}_history_orders']=np.log1p(n).astype('float32')
        out[f'{name}_history_age']=age.astype('float32')
        out[f'{name}_history_concentration']=(cnt/np.maximum(n,1)).astype('float32')
    return out

EFFECT_COLS=[f'{n}_{s}' for n in ENTITY for s in
             ('residual_effect','history_orders','history_age','history_concentration')]

def correction_fit(origin, anchor_trees):
    """Earlier six-month frozen snapshots supply OOF anchor scores and corrections."""
    snapshots=[]
    outer=pd.Timestamp(origin)
    for snap in pd.date_range('2023-07-01',outer-pd.DateOffset(months=6),freq='6MS'):
        fit=train.loc[train._order_time<snap]
        valid=train.loc[(train._order_time>=snap)&(train._order_time<snap+pd.DateOffset(months=6))]
        if len(fit)<25000 or len(valid)<1000: continue
        base,_,_=anchor(fit,valid,iterations=anchor_trees)
        v=residual_features(train,valid,snap)
        v['_anchor_logit']=np.log(np.clip(base,1e-5,1-1e-5)/np.clip(1-base,1e-5,1))
        snapshots.append(v)
    if not snapshots: return None
    fit=pd.concat(snapshots,ignore_index=True)
    cols=['_anchor_logit']+[c for c in EFFECT_COLS if c in fit]
    x=fit[cols].replace([np.inf,-np.inf],np.nan).fillna(0)
    m=LogisticRegression(C=.05,max_iter=300,solver='lbfgs')
    m.fit(x,fit.returned)
    return m,cols

ANCHOR_TREES=1800  # Fixed before outer assessment; tune only on an earlier inner fold if changed.
results={}; all_folds=list(folds())
for origin,fit,valid in all_folds:
    base,_,base_cols=anchor(fit,valid,ANCHOR_TREES)
    frozen=residual_features(train,valid,origin)
    learned=correction_fit(origin,ANCHOR_TREES)
    if learned:
        model,cols=learned
        frozen['_anchor_logit']=np.log(np.clip(base,1e-5,1-1e-5)/np.clip(1-base,1e-5,1))
        adjusted=model.predict_proba(frozen[cols].replace([np.inf,-np.inf],np.nan).fillna(0))[:,1]
        # Bound the correction in logit space to avoid large cohort extrapolation.
        adjusted_logit=np.log(np.clip(adjusted,1e-5,1-1e-5)/np.clip(1-adjusted,1e-5,1))
        corrected=1/(1+np.exp(-np.clip(frozen._anchor_logit+np.clip(adjusted_logit-frozen._anchor_logit,-.6,.6),-20,20)))
    else: corrected=base.copy()
    results[origin]={'anchor':diagnostics(valid,base),'adjusted':diagnostics(valid,corrected),
                     'paired_order_ci':paired_order_ci(valid,base,corrected,n=100)}
    assert results[origin]['anchor']['pooled'] < .99, 'Suspiciously perfect anchor AUC: audit feature leakage'
    save_prediction(f'adjusted_{origin[:7]}_anchor.csv',valid,base)
    save_prediction(f'adjusted_{origin[:7]}_candidate.csv',valid,corrected)
    print(origin,results[origin]['anchor']['pooled'],results[origin]['adjusted']['pooled'])
    del fit,valid,base,corrected
    gc.collect()

# Select only from earlier forecasts. July-December 2024 remains a development diagnostic.
earlier=[k for k in results if k<'2024-07-01']
gains=[results[k]['adjusted']['pooled']-results[k]['anchor']['pooled'] for k in earlier]
selected=bool(len(gains)>=2 and np.median(gains)>=.001 and gains[-1]>0)
final_base,anchor_model,_=anchor(train,test,ANCHOR_TREES)
anchor_model.save_model(str(OUT/'adjusted_history_anchor.cbm'))
final_pred=final_base.copy()
if selected:
    learned=correction_fit('2025-01-01',ANCHOR_TREES)
    if learned:
        model,cols=learned
        import joblib
        joblib.dump({'model':model,'columns':cols},OUT/'adjusted_history_correction.joblib')
        final=residual_features(train,test,'2025-01-01')
        logit=np.log(np.clip(final_base,1e-5,1-1e-5)/np.clip(1-final_base,1e-5,1))
        final['_anchor_logit']=logit
        p=model.predict_proba(final[cols].replace([np.inf,-np.inf],np.nan).fillna(0))[:,1]
        q=np.log(np.clip(p,1e-5,1-1e-5)/np.clip(1-p,1e-5,1))
        final_pred=1/(1+np.exp(-np.clip(logit+np.clip(q-logit,-.6,.6),-20,20)))
submission('submission_adjusted_history.csv',final_pred)
report('adjusted_history_report.json',{'folds':results,'selected':selected,
    'history_delay_days':75,'history_window_days':365,'anchor_trees':ANCHOR_TREES,
    'features':base_cols+EFFECT_COLS})
